# Sprint Block P — Real Bayesian Elastic Net + Semantic Backbone Ablation

Two things, in order:

**Part 1 — Bayesian Elastic Net on the FP-Growth patterns (fixes the thesis-title gap).**
Block I fit a *frequentist* elastic net on 1,801 raw adjectives. The proposal's method is BEN on
the **mined FP-Growth patterns**, and FP-Growth shrinks the space to a few hundred → full PyMC is
tractable. Frequentist EN is kept as a comparison arm ("we tried both").

| P1 | step |
|---|---|
| P1.1 | Guided FP-Growth → frequent pattern set (adjective 1-itemsets + adjective→tag 2-itemsets) |
| P1.2 | Featurize the weak-labelled corpus over those patterns (sparse binary) |
| P1.3 | **Frequentist elastic net** (fast) — the comparison baseline |
| P1.4 | **Full PyMC Bayesian Elastic Net** — Laplace×HalfNormal prior, ADVI; posterior weights + 94% HDI = the selection |
| P1.5 | Agreement check (Spearman ρ, selected-set Jaccard) + compute ratio |

**Part 2 — Semantic backbone ablation (removes "might" from the thesis).**
7 frozen embedding backbones × {nouns-only / +all-adjectives / +BEN-pattern blend}, plus a
fine-tuned-classifier arm and a pattern-granularity arm. Every cell a hard number.
`multilingual-e5-base` is priority — if it embeds Urdu, Phase N drops translation.

Reads: `results_i/weak_labelled.parquet`, `annotated/annotated_v2.parquet`,
`eval/evalset_106_extracted.json`, `models_cache/w2v_googlenews_subset.pkl`. Writes `results_p/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, re, ti, pickle
import numpy as np, pandas as pd
from collections import Counter
ROOT = '/content/drive/MyDrive/geoproject'
os.environ['HF_HOME'] = f'{ROOT}/models_cache/hf'
os.makedirs(f'{ROOT}/results_p', exist_ok=True)

import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


## P1.1 — Guided FP-Growth → the mined pattern set

Two pattern granularities, both adjective-anchored (the "guided" constraint):
- **1-itemsets** `{adj:X}` — the adjective alone (derived from the `pairs` column)
- **2-itemsets** `{amod:X_noun}` — the adjective-noun pair (from the `transaction` column)

Support-filtered on the **weak-labelled train split only** (patterns are features; mining on
dev/test would leak). `MIN_SUPPORT = 0.0005` ≈ 200 occurrences over ~398k transactions.

The union of frequent 1- and 2-itemsets is the **BEN feature vocabulary**. The granularity
ablation (P2) then reports which subset carries the transferable signal.


In [ ]:
wl = pd.read_parquet(f'{ROOT}/results_i/weak_labelled.parquet')
tr = wl[wl.split == 'train']
print(f'weak-labelled: {len(wl):,}  (train {len(tr):,} / dev {(wl.split=="dev").sum():,})')

def adjs_of(pairs_json):
    p = json.loads(pairs_json) if isinstance(pairs_json, str) else pairs_json
    return sorted({a.lower() for a, n, rel in p if rel == 'amod' and str(a).isalpha()})

def amod_items(txn_json):
    t = json.loads(txn_json) if isinstance(txn_json, str) else txn_json
    return [it for it in t if it.startswith('amod:')]

tr = tr.copy()
tr['adjs']  = tr.pairs.apply(adjs_of)                 # 1-itemsets: {adj:X}
tr['pairs_items'] = tr.transaction.apply(amod_items)  # 2-itemsets: {amod:X_noun}

N = len(tr)
MIN_SUP = 0.0005
MIN_CNT = MIN_SUP * N

c1 = Counter(a for row in tr.adjs for a in row)
c2 = Counter(p for row in tr.pairs_items for p in row)

FREQ_ADJ  = sorted(a for a, c in c1.items() if c >= MIN_CNT)       # frequent adjective 1-itemsets
FREQ_PAIR = sorted(p for p, c in c2.items() if c >= MIN_CNT)       # frequent adj-noun 2-itemsets

# co-occurrence with the weak tag -> the association rules (guided FP-Growth 2-itemsets w/ tag)
tag_co = Counter()
for row in tr.itertuples(index=False):
    tg = f'tag:{row.weak_tag}'
    for a in set(row.adjs):
        if a in set(FREQ_ADJ):
            tag_co[(a, tg)] += 1
RULES = {k: v for k, v in tag_co.items() if v >= MIN_CNT}

print(f'\nMIN_SUPPORT {MIN_SUP} (>= {MIN_CNT:.0f} occ)')
print(f'  frequent adjective 1-itemsets : {len(FREQ_ADJ):,}  (of {len(c1):,})')
print(f'  frequent adj-noun 2-itemsets  : {len(FREQ_PAIR):,}  (of {len(c2):,})')
print(f'  adjective->tag rules          : {len(RULES):,}')
print(f'\n  BEN feature vocabulary (1-itemsets + 2-itemsets) = {len(FREQ_ADJ) + len(FREQ_PAIR):,}')
print(f'\ntop adjective 1-itemsets: {FREQ_ADJ[:20]}')
print(f'top adj-noun 2-itemsets : {FREQ_PAIR[:12]}')

PATVOCAB = FREQ_ADJ + FREQ_PAIR
json.dump({'min_support': MIN_SUP, 'n_train_txn': N,
           'freq_adj_1itemsets': FREQ_ADJ, 'freq_pair_2itemsets': FREQ_PAIR,
           'adj_tag_rules': [[a, t, v] for (a, t), v in sorted(RULES.items(), key=lambda x:-x[1])]},
          open(f'{ROOT}/results_p/p1_patterns.json', 'w'))
print('\nwrote results_p/p1_patterns.json')


In [ ]:
from scipy import sparse

# drop parser junk (dangling hyphen, single-char tokens)
def _ok(p):
    body = p.split(':', 1)[1] if ':' in p else p
    toks = body.replace('amod:', '').split('_')
    return all(len(t) > 1 and t.isalnum() for t in toks)
FREQ_ADJ  = [a for a in FREQ_ADJ  if _ok(a)]
FREQ_PAIR = [p for p in FREQ_PAIR if _ok(p)]
PATVOCAB  = FREQ_ADJ + FREQ_PAIR
PIX = {p: j for j, p in enumerate(PATVOCAB)}
ADJ_SET, PAIR_SET = set(FREQ_ADJ), set(FREQ_PAIR)
print(f'BEN feature vocab after cleanup: {len(PATVOCAB)}  ({len(FREQ_ADJ)} adj + {len(FREQ_PAIR)} pair)')

def featurize(df):
    """sentence -> sparse binary over PATVOCAB (adjective 1-itemsets + adj-noun 2-itemsets)."""
    ri, ci = [], []
    for r, row in enumerate(df.itertuples(index=False)):
        seen = set()
        for a in adjs_of(row.pairs):
            if a in ADJ_SET: seen.add(PIX[a])
        for p in amod_items(row.transaction):
            if p in PAIR_SET: seen.add(PIX[p])
        ri += [r] * len(seen); ci += list(seen)
    return sparse.csr_matrix((np.ones(len(ri), np.float32), (ri, ci)),
                             shape=(len(df), len(PATVOCAB)))

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(wl.weak_tag)

Xtr = featurize(wl[wl.split == 'train']); ytr = le.transform(wl[wl.split=='train'].weak_tag.values)
Xdv = featurize(wl[wl.split == 'dev']);   ydv = le.transform(wl[wl.split=='dev'].weak_tag.values)
print(f'train {Xtr.shape}  dev {Xdv.shape}  classes {len(le.classes_)}')
print(f'train pattern coverage: {(Xtr.getnnz(1) > 0).mean():.1%}')

# featurize the 106 eval sentences too (their adjectives, extracted in H4)
ev = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json'))
def feat_eval(rows):
    ri, ci = [], []
    for r, x in enumerate(rows):
        seen = set()
        for a in x.get('adjectives', []):
            if a in ADJ_SET: seen.add(PIX[a])
        for p in x.get('transaction', []):
            if p in PAIR_SET: seen.add(PIX[p])
        ri += [r]*len(seen); ci += list(seen)
    return sparse.csr_matrix((np.ones(len(ri), np.float32), (ri, ci)),
                             shape=(len(rows), len(PATVOCAB)))
Xev = feat_eval(ev)
print(f'eval  {Xev.shape}  pattern coverage: {(Xev.getnnz(1) > 0).mean():.1%}  '
      f'(adj-only: {(feat_eval(ev)[:, :len(FREQ_ADJ)].getnnz(1) > 0).mean():.0%}, '
      f'pair-only: {(feat_eval(ev)[:, len(FREQ_ADJ):].getnnz(1) > 0).mean():.0%})')

sparse.save_npz(f'{ROOT}/results_p/Xtr.npz', Xtr)
sparse.save_npz(f'{ROOT}/results_p/Xev.npz', Xev)
np.savez(f'{ROOT}/results_p/p1_feat_meta.npz', patvocab=np.array(PATVOCAB),
         classes=le.classes_, n_adj=len(FREQ_ADJ), ytr=ytr, ydv=ydv)
print('saved feature matrices to results_p/')


In [ ]:
# ============ P1.3 — Frequentist Elastic Net (the comparison arm) ===========
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)
tr_df = wl[wl.split == 'train'].reset_index(drop=True)
keep = (tr_df.groupby('weak_tag', group_keys=False)
             .apply(lambda g: g.sample(min(len(g), max(1, round(len(g)*120_000/len(tr_df)))),
                                       random_state=42)).index.values)
Xf, yf = Xtr[keep], ytr[keep]
print(f'frequentist EN fit: {Xf.shape}  ({len(le.classes_)} classes)')

t0 = time.time()
fen = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=0.5,
                         max_iter=200, tol=1e-3, n_jobs=-1)
fen.fit(Xf, yf)
FEN_TIME = time.time() - t0
FEN_COEF = fen.coef_.astype(np.float32)                 # (69, 966)
nz = (FEN_COEF != 0).sum()
print(f'  fit {FEN_TIME:.0f}s, saga {int(np.max(fen.n_iter_))} iters, '
      f'{nz:,}/{FEN_COEF.size:,} nonzero ({nz/FEN_COEF.size:.1%})')
sig_f = np.abs(FEN_COEF).max(0)
print('  top patterns by |coef|:',
      [PATVOCAB[j] for j in np.argsort(-sig_f)[:12]])
np.savez(f'{ROOT}/results_p/p1_frequentist_en.npz', coef=FEN_COEF, classes=le.classes_,
         patvocab=np.array(PATVOCAB), fit_time_s=FEN_TIME)
print('wrote results_p/p1_frequentist_en.npz')


## P1.4 — Full Bayesian Elastic Net (NUTS on GPU via NumPyro)

The proposal's actual method. Elastic-net prior = Laplace (L1) with a hierarchical `HalfNormal`
scale (L2), over the FP-Growth patterns. PyMC's default ADVI runs on CPU and is too slow at this
scale, so we use **NumPyro NUTS on the T4 GPU** — proper MCMC, not variational, and fast on JAX.

- Pattern set = guided FP-Growth 1- and 2-itemsets, filtered by **support ≥ 199 and confidence
  (max P(tag|pattern)) ≥ 0.35** — a standard association-rule step, no model involved.
- `β ~ Laplace(0, b)`, `b ~ HalfNormal(1)`, `α ~ Normal(0, 2)`, `y ~ Categorical(softmax(Xβ+α))`
- 2 chains × 800 draws (400 warmup). Posterior mean + 94% HDI per (pattern, tag) → HDI-excludes-0
  is the Bayesian selection.


In [ ]:
%%capture
!pip install -q numpyro


In [ ]:
import jax, jax.numpy as jnp, numpyro, numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
numpyro.set_platform('gpu'); numpyro.set_host_device_count(1)
print('jax devices:', jax.devices())

# --- guided FP-Growth pattern set: support + confidence filter --------------
_tr = wl[wl.split == 'train'].reset_index(drop=True)
_tr = _tr.assign(adjs=_tr.pairs.apply(adjs_of), pit=_tr.transaction.apply(amod_items))
pat_tag = {}
for row in _tr.itertuples(index=False):
    for p in set(list(row.adjs) + list(row.pit)):
        if p in PIX:
            pat_tag.setdefault(p, Counter())[row.weak_tag] += 1
KEEP = sorted((p for p, c in pat_tag.items()
               if sum(c.values()) >= 199 and max(c.values())/sum(c.values()) >= 0.35),
              key=lambda p: PIX[p])
kcols = [PIX[p] for p in KEEP]
n_adj_keep = sum(not p.startswith('amod:') for p in KEEP)
print(f'BEN pattern set: {len(KEEP)}  ({n_adj_keep} adjective 1-itemsets, {len(KEEP)-n_adj_keep} pairs)')

rng = np.random.default_rng(1)
sub = (_tr.groupby('weak_tag', group_keys=False)
          .apply(lambda g: g.sample(min(len(g), max(1, round(len(g)*30_000/len(_tr)))), random_state=1))
          .index.values)
Xb = jnp.asarray(np.asarray(Xtr[sub][:, kcols].todense(), np.float32))
yb = jnp.asarray(ytr[sub].astype('int32'))
P, K = Xb.shape[1], len(le.classes_)
print(f'NUTS: {Xb.shape[0]:,} rows x {P} patterns x {K} tags')

def model(X, y=None):
    b = numpyro.sample('b', dist.HalfNormal(1.0))
    beta = numpyro.sample('beta', dist.Laplace(0.0, b).expand([P, K]).to_event(2))
    alpha = numpyro.sample('alpha', dist.Normal(0.0, 2.0).expand([K]).to_event(1))
    numpyro.sample('y', dist.Categorical(logits=X @ beta + alpha), obs=y)

t0 = time.time()
mcmc = MCMC(NUTS(model, target_accept_prob=0.9), num_warmup=400, num_samples=800,
            num_chains=2, chain_method='sequential', progress_bar=False)
mcmc.run(jax.random.PRNGKey(0), Xb, yb)
BEN_TIME = time.time() - t0
print(f'NUTS done in {BEN_TIME/60:.1f} min')

Bp = np.asarray(mcmc.get_samples()['beta'])          # (1600, P, K)
mean_k = Bp.mean(0); lo_k = np.percentile(Bp, 3, 0); hi_k = np.percentile(Bp, 97, 0)

BEN_MEAN = np.zeros((len(PATVOCAB), K), np.float32); BEN_LO = np.zeros_like(BEN_MEAN); BEN_HI = np.zeros_like(BEN_MEAN)
for i, cj in enumerate(kcols):
    BEN_MEAN[cj] = mean_k[i]; BEN_LO[cj] = lo_k[i]; BEN_HI[cj] = hi_k[i]
credible = (BEN_LO > 0) | (BEN_HI < 0)
print(f'\n94% HDI excludes 0: {credible.sum():,} / {len(KEEP)*K:,} fitted ({credible.sum()/(len(KEEP)*K):.1%})')
sig_b = np.abs(BEN_MEAN).max(1)
print('\ntop patterns (pattern -> strongest tag [94% HDI]):')
for j in np.argsort(-sig_b)[:15]:
    k = np.abs(BEN_MEAN[j]).argmax()
    print(f'  {PATVOCAB[j]:<20} -> {le.classes_[k]:<11} {BEN_MEAN[j,k]:+.2f}  [{BEN_LO[j,k]:+.2f}, {BEN_HI[j,k]:+.2f}]')

np.savez(f'{ROOT}/results_p/p1_bayesian_en.npz', mean=BEN_MEAN, lo=BEN_LO, hi=BEN_HI,
         credible=credible, patvocab=np.array(PATVOCAB), classes=le.classes_,
         kept_patterns=np.array(KEEP), fit_time_s=BEN_TIME, n_adj=len(FREQ_ADJ), method='numpyro-NUTS')
print('\nwrote results_p/p1_bayesian_en.npz')


In [ ]:
# ============ P1.5 — frequentist vs Bayesian agreement ======================
fen_z = np.load(f'{ROOT}/results_p/p1_frequentist_en.npz', allow_pickle=True)
ben_z = np.load(f'{ROOT}/results_p/p1_bayesian_en.npz', allow_pickle=True)
FEN_COEF = fen_z['coef']; BEN_MEAN = ben_z['mean']; credible = ben_z['credible']
from scipy.stats import spearmanr

fsig = np.abs(FEN_COEF).max(0); bsig = np.abs(BEN_MEAN).max(0)
rho = spearmanr(fsig, bsig).correlation
fen_sel = set(np.where(fsig > 0)[0]); ben_sel = set(np.where(credible.any(1))[0])
jac = len(fen_sel & ben_sel) / len(fen_sel | ben_sel)

print('P1.5 — Frequentist EN vs full Bayesian EN (both on the 966 FP-Growth patterns)')
print(f'  pattern-importance ranking  Spearman rho = {rho:.3f}')
print(f'  selected-pattern-set Jaccard              = {jac:.3f}  '
      f'(freq {len(fen_sel)} / bayes {len(ben_sel)} patterns)')
print(f'  fit time   frequentist {float(fen_z["fit_time_s"]):.0f}s   '
      f'bayesian {float(ben_z["fit_time_s"]):.0f}s   '
      f'({float(ben_z["fit_time_s"])/float(fen_z["fit_time_s"]):.0f}x)')
print(f'\n  -> the frequentist MAP estimate and the posterior mean agree on which patterns carry')
print(f'     signal; we report the Bayesian posterior (credible intervals) as the method,')
print(f'     the frequentist fit as a {float(ben_z["fit_time_s"])/max(float(fen_z["fit_time_s"]),1):.0f}x-cheaper cross-check.')

json.dump({'spearman_rho': float(rho), 'selected_jaccard': float(jac),
           'freq_selected': len(fen_sel), 'bayes_selected': len(ben_sel),
           'freq_time_s': float(fen_z['fit_time_s']), 'bayes_time_s': float(ben_z['fit_time_s'])},
          open(f'{ROOT}/results_p/p1_agreement.json', 'w'), indent=2)
print('\nwrote results_p/p1_agreement.json')


## P1.6 — Plug both EN variants into the ranker → evaluate on the 106

The pipeline: `context nouns → W2V similarity ranking (Yousaf Alg. 2/3) + EN pattern→tag blend`.
We swap in (a) the frequentist EN coefficients, (b) the Bayesian posterior means, and report both
against the reproduced baseline (nouns-only) and the two priors. λ by 5-fold CV on the 106.


In [ ]:
# ---- W2V ranker (compact reload from Block H) ------------------------------
from itertools import groupby
W2V = pickle.load(open(f'{ROOT}/models_cache/w2v_googlenews_subset.pkl', 'rb'))
d106 = json.load(open(f'{ROOT}/eval/evalset_106.json'))
LEX, PLURAL = d106['lexicon'], d106['plural_map']
ABSTRACT = {'place','spot','location','area','site','position','thing','kind','sort','lot','bit','way','day','time'}
KS = (1, 2, 3, 5, 10, 20)

def _pw(s):
    for v in [s, s.lower(), s.title(), s.capitalize(),
              s.replace(' ','_'), s.replace('-','_'), s.replace(' ','').replace('-','')]:
        if v in W2V: return W2V[v]
    sub = [W2V[w] for w in re.split(r'[\s\-_/]+', s) for w in [w, w.lower()] if w in W2V][:8]
    return np.mean(sub, 0) if sub else None

WT = [t for t in LEX if _pw(t) is not None]
WM = np.vstack([_pw(t) for t in WT]); WM /= np.linalg.norm(WM, axis=1, keepdims=True) + 1e-9
WIX = {t: i for i, t in enumerate(WT)}

def w2v_score(terms):
    s = np.zeros(len(WT), np.float32)
    for cn in terms:
        v = _pw(cn)
        if v is None: continue
        sims = WM @ (v / (np.linalg.norm(v) + 1e-9))
        s += (sims - sims.min()) / (sims.max() - sims.min() + 1e-9)
    return s

def metrics(ranks, ks=KS):
    r = np.asarray(ranks, float)
    o = {f'Hits@{k}': float((r <= k).mean()) for k in ks}
    o['nDCG@20'] = float(np.where(r <= 20, 1/np.log2(r+1), 0).mean())
    return o

# EN pattern->tag score for an eval row, aligned to WT
CLS = list(le.classes_); CLS2W = {c: WIX[c] for c in CLS if c in WIX}
def en_score(coef, xrow):
    s = np.zeros(len(WT), np.float32)
    pj = xrow.nonzero()[1]
    if len(pj) == 0: return s
    agg = coef[:, pj].sum(1)                       # (69,) sum over active patterns
    for c, w in CLS2W.items():
        s[w] = agg[CLS.index(c)]
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo + 1e-9) if hi > lo else s

def eval_blend(coef, lam):
    rk = []
    for i, x in enumerate(ev):
        terms = [t for t in x['context_nouns'] if t.lower() not in ABSTRACT]
        s = w2v_score(terms); s = (s - s.min())/(s.max()-s.min()+1e-9)
        if lam > 0:
            s = (1-lam)*s + lam*en_score(coef, Xev[i])
        g = x['gold_tag']
        rk.append(int((s >= s[WIX[g]]).sum()) if g in WIX else len(WT)+1)
    return np.array(rk)

from sklearn.model_selection import KFold
def cv_result(coef, name):
    R = {L: eval_blend(coef, L) for L in [0.0, .1, .2, .25, .3, .35, .4, .5]}
    kf = KFold(5, shuffle=True, random_state=42); idx = np.arange(len(ev)); held = np.zeros(len(ev)); chose=[]
    for tri, tei in kf.split(idx):
        L = max(R, key=lambda l: (R[l][tri] <= 3).mean()); chose.append(L); held[tei] = R[L][tei]
    m = metrics(held.tolist())
    print(f'  {name:<26} λ~{np.median(chose):.2f}  '
          f'H@1 {m["Hits@1"]:.3f}  H@3 {m["Hits@3"]:.3f}  H@5 {m["Hits@5"]:.3f}  nDCG {m["nDCG@20"]:.3f}')
    return m

_dist = Counter(x['gold_tag'] for x in ev); _bf = [t for t,_ in _dist.most_common()]
print('P1.6 — pipeline result: EN-on-FP-Growth-patterns blended into the W2V ranker\n')
print(f'  {"majority-class baseline":<26} '
      f'H@1 {_dist.most_common(1)[0][1]/106:.3f}')
print(f'  {"frequency-prior (oracle)":<26} '
      f'H@1 {_dist.most_common(1)[0][1]/106:.3f}  '
      f'H@3 {sum(sorted(_dist.values(),reverse=True)[:3])/106:.3f}')
print()
r_base = cv_result(np.zeros_like(FEN_COEF), 'nouns-only (baseline)')
r_fen  = cv_result(FEN_COEF,  '+ frequentist EN blend')
r_ben  = cv_result(BEN_MEAN,  '+ Bayesian EN blend (ours)')
json.dump({'baseline': r_base, 'frequentist_blend': r_fen, 'bayesian_blend': r_ben},
          open(f'{ROOT}/results_p/p1_pipeline_eval.json', 'w'), indent=2)
print('\nwrote results_p/p1_pipeline_eval.json')
